# Support Vector Machine (SVM)
Support Vector Machines (SVM) zijn krachtige en veelzijdige algoritmen voor zowel classificatie- als regressietaken. Het belangrijkste idee achter SVM is het vinden van een hypervlak dat de gegevenspunten van verschillende klassen in de feature-ruimte scheidt met de grootste marge. Dit hypervlak wordt bepaald door de zogenaamde "support vectors", de gegevenspunten die het dichtst bij het hypervlak liggen. SVM kan ook worden uitgebreid naar niet-lineaire classificatie door gebruik te maken van kernel-tricks, die de gegevens in een hogere dimensionale ruimte projecteren waar een lineaire scheiding mogelijk is. SVM is bijzonder effectief bij hoge-dimensionale datasets en wordt vaak gebruikt in toepassingen zoals tekstclassificatie en beeldherkenning.

# Uitgebreide Support Vector Machine Analyse
Dit onderzoek evalueert SVM prestatie over verschillende configuraties:
- **Kernel Types**: Linear, RBF, Polynomial, Sigmoid kernels
- **Feature Representaties**: Original features, PCA-reduced, LDA-reduced
- **Hyperparameter Optimalisatie**: GridSearchCV voor optimale parameters
- **SVM-Specifieke Analyse**: Support vectors, decision boundaries, kernel performance
- **Interpreteerbaarheid**: Feature importance door kernel analyse en support vector examinatie

In [1]:
import sys
import os
import warnings
import pandas as pd
warnings.filterwarnings('ignore')

# Add the parent directory (project root) to Python path
project_root = os.path.abspath('..')  # Go up one level from current notebook
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from utils.libs_generic_functions import LibsDataLoader, LibsDataPreprocessor
from utils.baseline_correction_functions import BaselineCorrector, analyze_correction_quality, print_quality_report
from utils.normalization_functions import Normalizer
from utils.pca_analysis_functions import PCAAnalyzer, quick_pca_analysis
from utils.lda_analysis_functions import quick_lda_analysis
print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
%%time
df = pd.read_csv("../data/csv/processed_data.csv")

In [ ]:
%%time
Preprocessor = LibsDataPreprocessor()
X_train, X_test, y_train, y_test, label_encoder, wavelength_columns = Preprocessor.prepare_ml_data(df)


PREPARING INDIVIDUAL MEASUREMENTS FOR ML
Available columns in df_individual: ['200.000', '200.098', '200.195', '200.293', '200.391', '200.489', '200.586', '200.684', '200.782', '200.879']...
Total columns: 8191
Meta columns found: ['tire_number', 'origin', 'measurement_id']
Wavelength features found: 8188
Feature matrix shape: (51453, 8188)
Target distribution:
origin
tread         17936
innerliner    17260
sidewall      16257
Name: count, dtype: int64

Data quality check:
Feature matrix shape: (51453, 8188)
Target distribution:
origin
tread         17936
innerliner    17260
sidewall      16257
Name: count, dtype: int64

Data quality check:
Missing values: 0
Missing values: 0
Infinite values: 0
Infinite values: 0

Final ML dataset shape: (51453, 8191)
Samples available: 51,453

Final ML dataset shape: (51453, 8191)
Samples available: 51,453

Train/Test split:
Training set: (41162, 8188)
Test set: (10291, 8188)

Training set distribution:
origin
tread         14349
innerliner    13808


In [ ]:
%%time
# Feature normalization 
print("Normalizing features...")
normaliser = Normalizer()
X_train_scaled = normaliser.apply_standard_normal_variate_on_dataset(X_train)
X_test_scaled = normaliser.apply_standard_normal_variate_on_dataset(X_test)
print("Data preprocessing completed successfully!")

# Display train/test distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Training set distribution
train_unique, train_counts = np.unique(y_train, return_counts=True)
ax1.bar(range(len(train_unique)), train_counts, alpha=0.7, color=['red', 'green', 'blue'])
ax1.set_title('Training Set Distribution')
ax1.set_xlabel('Class Index')
ax1.set_ylabel('Number of Samples')
ax1.set_xticks(range(len(train_unique)))
ax1.set_xticklabels(("tread", "innerliner", "sidewall"), rotation=45)

# Test set distribution  
test_unique, test_counts = np.unique(y_test, return_counts=True)
ax2.bar(range(len(test_unique)), test_counts, alpha=0.7, color=['red', 'green', 'blue'])
ax2.set_title('Test Set Distribution')
ax2.set_xlabel('Class Index')
ax2.set_ylabel('Number of Samples')
ax2.set_xticks(range(len(test_unique)))
ax2.set_xticklabels(("tread", "innerliner", "sidewall"), rotation=45)

plt.tight_layout()
plt.show()

## 1. SVM op originele features

In [ ]:
%%time

# 2. SVM Analysis on Original Features with Multiple Kernels
print("=" * 60)
print("2. SVM ANALYSIS ON ORIGINAL FEATURES")
print("=" * 60)

# Define comprehensive SVM configurations
svm_configs = {
    'Linear': {
        'kernel': 'linear',
        'param_grid': {
            'C': [0.01, 0.1, 1, 10, 100, 1000]
        }
    },
    'RBF': {
        'kernel': 'rbf', 
        'param_grid': {
            'C': [0.1, 1, 10, 100, 1000],
            'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
        }
    },
    'Polynomial': {
        'kernel': 'poly',
        'param_grid': {
            'C': [0.1, 1, 10, 100],
            'degree': [2, 3, 4],
            'gamma': ['scale', 'auto', 0.01, 0.1]
        }
    },
    'Sigmoid': {
        'kernel': 'sigmoid',
        'param_grid': {
            'C': [0.1, 1, 10, 100],
            'gamma': ['scale', 'auto', 0.01, 0.1, 1]
        }
    }
}

# Cross-validation setup
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store results for each kernel 
original_results = {}
best_original_result = {'accuracy': 0, 'kernel': '', 'params': {}}

print("Testing SVM kernels on original features...")
print("This may take several minutes due to high dimensionality...")


for kernel_name, config in svm_configs.items():
    print(f"  {kernel_name} kernel...", end=" ")
    start_time = time.time()
    
    try:
        # Create SVM with specific kernel
        svm = SVC(kernel=config['kernel'], random_state=42, probability=True)
        
        # Grid search for optimal parameters
        grid_search = GridSearchCV(
            svm, 
            config['param_grid'], 
            cv=cv_folds, 
            scoring='accuracy',
            n_jobs=-1,
            verbose=0
        )
        
        # Fit and evaluate
        grid_search.fit(X_train_scaled, y_train)
        
        # Test set performance
        test_accuracy = grid_search.score(X_test_scaled, y_test)
        
        # Store results
        result = {
            'best_score': grid_search.best_score_,
            'test_accuracy': test_accuracy,
            'best_params': grid_search.best_params_,
            'n_support_vectors': grid_search.best_estimator_.n_support_.sum(),
            'support_vector_ratio': grid_search.best_estimator_.n_support_.sum() / len(X_train_scaled),
            'training_time': time.time() - start_time,
            'best_estimator': grid_search.best_estimator_
        }
        
        original_result[kernel_name] = result
        
        # Track best overall result
        if test_accuracy > best_original_result['accuracy']:
            best_original_result = {
                'accuracy': test_accuracy,
                'kernel': kernel_name,
                'params': grid_search.best_params_,
                'estimator': grid_search.best_estimator_
            }
        
        print(f"CV: {grid_search.best_score_:.4f}, Test: {test_accuracy:.4f} ({time.time()-start_time:.1f}s)")
            
        except Exception as e:
            print(f"Error: {str(e)}")
            original_results[kernel_name] = {
                'error': str(e),
                'test_accuracy': 0,
                'best_score': 0
            }

print(f"\n🏆 Best Original Features Result:")
print(f"   Kernel: {best_original_result['kernel']}")
print(f"   Test Accuracy: {best_original_result['accuracy']:.4f}")
print(f"   Parameters: {best_original_result['params']}")

# Create comprehensive results visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# 1. Accuracy heatmap by kernel
accuracy_matrix = []
kernel_names = list(svm_configs.keys())

for kernel_name in kernel_names:
    if kernel_name in original_results and 'test_accuracy' in original_results[kernel_name]:
        accuracy_matrix.append(original_results[kernel_name]['test_accuracy'])
    else:
        accuracy_matrix.append(0)

accuracy_df = pd.DataFrame(accuracy_matrix, index=['Original'], columns=kernel_names)
sns.heatmap(accuracy_df, annot=True, fmt='.4f', cmap='viridis', ax=ax1)
ax1.set_title('SVM Test Accuracy by Kernel', fontsize=14, fontweight='bold')
ax1.set_xlabel('Kernel Type')


# 2. Support vector ratios
sv_ratios = []
labels = []
for kernel_name in kernel_names:
    if (kernel_name in original_results and 
        'support_vector_ratio' in original_results[kernel_name]):
        sv_ratios.append(original_results[kernel_name]['support_vector_ratio'])
        labels.append(f"{kernel_name} (Original)")

bars = ax2.bar(range(len(sv_ratios)), sv_ratios, alpha=0.7)
ax2.set_title('Support Vector Ratios by Configuration', fontsize=14, fontweight='bold')
ax2.set_ylabel('Support Vector Ratio')
ax2.set_xlabel('Kernel')
ax2.set_xticks(range(len(labels)))
ax2.set_xticklabels(labels, rotation=45, ha='right')
ax2.grid(axis='y', alpha=0.3)

# Highlight best performing bar
best_idx = np.argmax([acc for row in accuracy_matrix for acc in row])
bars[best_idx].set_color('red')
bars[best_idx].set_alpha(0.9)

# 3. Training time comparison
training_times = []
for kernel_name in kernel_names:
    if (kernel_name in original_results and 
        'training_time' in original_results[kernel_name]):
        training_times.append(original_results[kernel_name]['training_time'])

ax3.bar(range(len(training_times)), training_times, alpha=0.7, color='orange')
ax3.set_title('Training Time by Configuration', fontsize=14, fontweight='bold')
ax3.set_ylabel('Training Time (seconds)')
ax3.set_xlabel('Kernel')
ax3.set_xticks(range(len(labels)))
ax3.set_xticklabels(labels, rotation=45, ha='right')
ax3.grid(axis='y', alpha=0.3)

# 4. Accuracy vs Training Time scatter
accuracies_flat = [acc for row in accuracy_matrix for acc in row]
ax4.scatter(training_times, accuracies_flat, s=100, alpha=0.7, c=range(len(training_times)), cmap='viridis')
ax4.set_xlabel('Training Time (seconds)')
ax4.set_ylabel('Test Accuracy')
ax4.set_title('Accuracy vs Training Time Trade-off', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

# Annotate best point
best_acc_idx = np.argmax(accuracies_flat)
ax4.annotate(f'Best: {labels[best_acc_idx]}', 
            xy=(training_times[best_acc_idx], accuracies_flat[best_acc_idx]),
            xytext=(10, 10), textcoords='offset points',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='red', alpha=0.3),
            arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

plt.tight_layout()
plt.show()

# Detailed classification report for best model
print(f"\n📊 Detailed Results for Best Model ({best_original_result['kernel']} + {best_original_result}):")
print("=" * 80)

y_pred = best_original_result['estimator'].predict(X_test)
y_pred_proba = best_original_result['estimator'].predict_proba(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

print(f"\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print(f"\nModel Details:")
print(f"- Support Vectors: {best_original_result['estimator'].n_support_.sum()} ({best_original_result['estimator'].n_support_.sum()/len(X_train_best)*100:.1f}% of training data)")
print(f"- Support Vectors per class: {dict(zip(class_names, best_original_result['estimator'].n_support_))}")
print(f"- Number of classes: {len(class_names)}")

print(f"\nOriginal features SVM analysis complete!")
print(f"Best configuration: {best_original_result['kernel']} kernel with {best_original_result}")
print(f"Achieved {best_original_result['accuracy']:.4f} accuracy on {X_train.shape[1]} features")

2. SVM ANALYSIS ON ORIGINAL FEATURES
Testing SVM kernels with different scalers on original features...
This may take several minutes due to high dimensionality...

--- Testing with StandardScaler ---
  Linear kernel... 

KeyboardInterrupt: 

## 2. SVM op PCA-gereduceerde features

In [ ]:
%%time

result = quick_pca_analysis(
        X_train_scaled, X_test_scaled, y_train, y_test,
        classifier_name='SVM', label_encoder=label_encoder, plot=True
    )
    
    # Safely unpack results
    if isinstance(result, tuple) and len(result) == 3:
        pca_analyzer, analysis_results, svm_results = result
        
        # Check if we have valid results
        if svm_results is not None and 'best_result' in svm_results:
            best_result = svm_results['best_result']
            
            if best_result is not None:
                print(f"\n🏆 Best SVM-PCA Configuration:")
                print(f"   Components: {best_result['n_components']}")
                print(f"   Accuracy: {best_result['accuracy']:.4f}")
                print(f"   Variance explained: {best_result['variance_explained']:.3f}")

                # Classification report for best result
                print(f"\nClassification Report (PCA - {best_result['n_components']} components):")
                print(classification_report(y_test, best_result['y_pred'], 
                                          target_names=label_encoder.classes_))

                # Plot confusion matrix for best result
                pca_analyzer.plot_confusion_matrix(
                    y_test, best_result['y_pred'], 'SVM',
                    best_result['n_components'], label_encoder
                )

3. SVM ANALYSIS WITH PCA DIMENSIONALITY REDUCTION
Testing SVM with PCA-reduced features...
Evaluating multiple PCA configurations and kernel combinations...

--- PCA Configuration: 95% Variance ---


KeyError: ''

## 3. KNN op LDA gereduceerde features

In [ ]:
%%time

# Quick LDA analysis for SVM
results = quick_lda_analysis(X_train_scaled, X_test_scaled, y_train, y_test, 
                           classifier_name='SVM', label_encoder=label_encoder)

# Extract results properly
best_lda_result = {
    'accuracy': results['best_result']['test_accuracy'],
    'estimator': results['best_result']['classifier'],
    'kernel': 'linear',  # Default for LDA analysis
    'n_components': results['best_result']['n_components'],
    'params': results['best_result']['optimal_params']
}

print(f"\n🏆 Best SVM-LDA Configuration:")
print(f"   Components: {best_lda_result['n_components']}")
print(f"   Accuracy: {best_lda_result['accuracy']:.4f}")
print(f"   Kernel: {best_lda_result['kernel']}")

# Classification report for best result
print(f"\nClassification Report (LDA - {best_lda_result['n_components']} components):")
print(classification_report(y_test, results['best_result']['y_pred'], 
                          target_names=label_encoder.classes_))

# Plot confusion matrix for best result
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, results['best_result']['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title(f'Confusion Matrix - SVM on LDA Features ({best_lda_result["n_components"]} components)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

4. SVM ANALYSIS WITH LDA DIMENSIONALITY REDUCTION


NameError: name 'y_train' is not defined

## 4. Resultaten Vergelijking

In [ ]:
%%time

# 5. SVM Interpretability and Support Vector Analysis
print("=" * 60)
print("5. SVM INTERPRETABILITY AND SUPPORT VECTOR ANALYSIS")
print("=" * 60)

# First, let's define the missing variables from previous analysis
# You'll need to extract these from your PCA and LDA results

# Get PCA results from previous analysis
pca_analyzer, pca_analysis_results, pca_performance_results = quick_pca_analysis(
    X_train_scaled, X_test_scaled, y_train, y_test,
    classifier_name='SVM', label_encoder=label_encoder, plot=False
)

best_pca_result = {
    'accuracy': pca_performance_results['best_result']['accuracy'],
    'estimator': pca_performance_results['best_result']['classifier'],
    'kernel': 'rbf',  # Default for SVM in PCA analysis
    'n_components': pca_performance_results['best_result']['n_components'],
    'params': {},
    'pca_transformer': pca_performance_results['best_result']['pca_model']
}

# Get LDA results
lda_results = quick_lda_analysis(X_train_scaled, X_test_scaled, y_train, y_test, 
                               classifier_name='SVM', label_encoder=label_encoder)

best_lda_result = {
    'accuracy': lda_results['best_result']['test_accuracy'],
    'estimator': lda_results['best_result']['classifier'],
    'kernel': 'linear',  # Default for LDA analysis
    'n_components': lda_results['best_result']['n_components'],
    'params': lda_results['best_result']['optimal_params']
}

max_lda_components = lda_results['max_possible_components']

# Transform data for LDA analysis
lda_model = lda_results['best_result']['lda_model']
X_train_lda = lda_model.fit_transform(X_train_scaled, y_train)
X_test_lda = lda_model.transform(X_test_scaled)

# Define class names
class_names = label_encoder.classes_

# Analyze the best-performing SVM models from each configuration
models_to_analyze = {
    'Original Features': {
        'model': best_original_result['estimator'],
        'X_train': X_train_original,
        'X_test': X_test_original,
        'kernel': best_original_result['kernel'],
        'features': X_train_original.shape[1],
        'config': f"{best_original_result['kernel']}"
    },
    'PCA Features': {
        'model': best_pca_result['estimator'],
        'X_train': best_pca_result['pca_transformer'].transform(X_train_original),
        'X_test': best_pca_result['pca_transformer'].transform(X_test_original),
        'kernel': best_pca_result['kernel'],
        'features': best_pca_result['n_components'],
        'config': f"{best_pca_result['kernel']} + PCA({best_pca_result['n_components']})"
    },
    'LDA Features': {
        'model': best_lda_result['estimator'],
        'X_train': X_train_lda,
        'X_test': X_test_lda,
        'kernel': best_lda_result['kernel'],
        'features': max_lda_components,
        'config': f"{best_lda_result['kernel']} + LDA({max_lda_components})"
    }
}

print("Analyzing support vector patterns and model interpretability...")

# Support vector analysis
support_vector_analysis = {}

for config_name, config in models_to_analyze.items():
    model = config['model']
    X_train_transformed = config['X_train']
    
    print(f"\n--- {config_name} Analysis ---")
    print(f"Configuration: {config['config']}")
    print(f"Kernel: {config['kernel']}")
    print(f"Features: {config['features']}")
    
    # Support vector statistics
    n_sv_total = model.n_support_.sum()
    n_sv_per_class = model.n_support_
    sv_ratio = n_sv_total / len(X_train_transformed)
    
    print(f"Support Vectors:")
    print(f"  Total: {n_sv_total} ({sv_ratio:.3f} of training data)")
    print(f"  Per class: {dict(zip(class_names, n_sv_per_class))}")
    
    # Decision function analysis
    decision_values = model.decision_function(config['X_test'])
    if len(class_names) == 2:
        # Binary classification
        margin_distances = np.abs(decision_values)
        confidence_scores = model.predict_proba(config['X_test']).max(axis=1)
    else:
        # Multi-class classification (one-vs-one)
        margin_distances = np.abs(decision_values).mean(axis=1)
        confidence_scores = model.predict_proba(config['X_test']).max(axis=1)
    
    print(f"Decision Analysis:")
    print(f"  Average margin distance: {margin_distances.mean():.4f}")
    print(f"  Margin std: {margin_distances.std():.4f}")
    print(f"  Average confidence: {confidence_scores.mean():.4f}")
    print(f"  Confidence std: {confidence_scores.std():.4f}")
    
    # Store for comparison
    support_vector_analysis[config_name] = {
        'n_support_vectors': n_sv_total,
        'support_vector_ratio': sv_ratio,
        'n_support_per_class': n_sv_per_class,
        'margin_distances': margin_distances,
        'confidence_scores': confidence_scores,
        'kernel': config['kernel'],
        'features': config['features']
    }

# Continue with the rest of your visualization code...
# (The plotting code remains the same as it was working correctly)

# Comprehensive visualization of SVM interpretability
fig = plt.figure(figsize=(20, 15))

# Create a 3x3 grid for visualizations
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Support vector comparison
ax1 = fig.add_subplot(gs[0, 0])
configs = list(support_vector_analysis.keys())
sv_counts = [support_vector_analysis[config]['n_support_vectors'] for config in configs]
sv_ratios = [support_vector_analysis[config]['support_vector_ratio'] for config in configs]

bars = ax1.bar(configs, sv_counts, alpha=0.7, color=['skyblue', 'lightgreen', 'orange'])
ax1.set_title('Support Vector Counts', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Support Vectors')
ax1.tick_params(axis='x', rotation=45)

for bar, count in zip(bars, sv_counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f'{count}', ha='center', va='bottom', fontweight='bold')

# 2. Support vector ratios
ax2 = fig.add_subplot(gs[0, 1])
bars = ax2.bar(configs, sv_ratios, alpha=0.7, color=['skyblue', 'lightgreen', 'orange'])
ax2.set_title('Support Vector Ratios', fontsize=12, fontweight='bold')
ax2.set_ylabel('Ratio of Training Data')
ax2.tick_params(axis='x', rotation=45)

for bar, ratio in zip(bars, sv_ratios):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{ratio:.3f}', ha='center', va='bottom', fontweight='bold')

# 3. Support vectors per class (stacked bar)
ax3 = fig.add_subplot(gs[0, 2])
class_sv_data = np.array([support_vector_analysis[config]['n_support_per_class'] 
                         for config in configs])

bottom = np.zeros(len(configs))
colors = ['red', 'green', 'blue']
for i, class_name in enumerate(class_names):
    ax3.bar(configs, class_sv_data[:, i], bottom=bottom, 
           label=class_name, alpha=0.7, color=colors[i])
    bottom += class_sv_data[:, i]

ax3.set_title('Support Vectors by Class', fontsize=12, fontweight='bold')
ax3.set_ylabel('Number of Support Vectors')
ax3.tick_params(axis='x', rotation=45)
ax3.legend()

# 4-6. Margin distance distributions
for i, config in enumerate(configs):
    ax = fig.add_subplot(gs[1, i])
    margins = support_vector_analysis[config]['margin_distances']
    
    ax.hist(margins, bins=30, alpha=0.7, color=['skyblue', 'lightgreen', 'orange'][i])
    ax.axvline(margins.mean(), color='red', linestyle='--', 
              label=f'Mean: {margins.mean():.3f}')
    ax.set_title(f'{config}\nMargin Distances', fontsize=11, fontweight='bold')
    ax.set_xlabel('Distance from Decision Boundary')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

# 7-9. Confidence score distributions
for i, config in enumerate(configs):
    ax = fig.add_subplot(gs[2, i])
    confidences = support_vector_analysis[config]['confidence_scores']
    
    ax.hist(confidences, bins=30, alpha=0.7, color=['skyblue', 'lightgreen', 'orange'][i])
    ax.axvline(confidences.mean(), color='red', linestyle='--', 
              label=f'Mean: {confidences.mean():.3f}')
    ax.set_title(f'{config}\nPrediction Confidence', fontsize=11, fontweight='bold')
    ax.set_xlabel('Confidence Score')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Rest of the analysis code continues as before...
print(f"\n🔍 KERNEL-SPECIFIC INTERPRETABILITY ANALYSIS:")
print("=" * 60)

for config_name, analysis in support_vector_analysis.items():
    kernel = analysis['kernel']
    print(f"\n{config_name} ({kernel} kernel):")
    
    if kernel == 'linear':
        print("  ✓ Linear kernel provides direct feature interpretability")
        print("  ✓ Decision boundary is a hyperplane in original space")
        print("  ✓ Feature weights can be extracted from model coefficients")
        print(f"  ✓ Uses {analysis['support_vector_ratio']:.1%} of training data as support vectors")
        
    elif kernel == 'rbf':
        print("  ✓ RBF kernel captures non-linear patterns")
        print("  ✓ Creates complex decision boundaries in high-dimensional space")
        print("  ✓ Support vectors define local decision regions")
        print(f"  ✓ Uses {analysis['support_vector_ratio']:.1%} of training data as support vectors")
        print("  ⚠ Less interpretable than linear kernel")
        
    elif kernel == 'poly':
        print("  ✓ Polynomial kernel captures polynomial interactions")
        print("  ✓ Decision boundary complexity depends on polynomial degree")
        print("  ✓ Can model feature interactions up to specified degree")
        print(f"  ✓ Uses {analysis['support_vector_ratio']:.1%} of training data as support vectors")
        print("  ⚠ Moderate interpretability")
        
    elif kernel == 'sigmoid':
        print("  ✓ Sigmoid kernel behaves like neural network activation")
        print("  ✓ Can approximate multi-layer perceptron behavior")
        print(f"  ✓ Uses {analysis['support_vector_ratio']:.1%} of training data as support vectors")
        print("  ⚠ Limited interpretability")

# Model complexity analysis
print(f"\n🎯 MODEL COMPLEXITY ANALYSIS:")
print("=" * 40)

complexity_scores = {}
for config_name, analysis in support_vector_analysis.items():
    # Complexity score based on support vector ratio and feature count
    sv_complexity = analysis['support_vector_ratio'] * 100  # More SVs = more complex
    feature_complexity = np.log10(analysis['features']) * 10  # More features = more complex
    
    # Kernel complexity multiplier
    kernel_multiplier = {
        'linear': 1.0,
        'rbf': 1.5,
        'poly': 1.3,
        'sigmoid': 1.4
    }
    
    total_complexity = (sv_complexity + feature_complexity) * kernel_multiplier.get(analysis['kernel'], 1.0)
    complexity_scores[config_name] = total_complexity
    
    print(f"{config_name}:")
    print(f"  Support Vector Complexity: {sv_complexity:.1f}")
    print(f"  Feature Complexity: {feature_complexity:.1f}")
    print(f"  Kernel Multiplier: {kernel_multiplier.get(analysis['kernel'], 1.0):.1f}")
    print(f"  Total Complexity Score: {total_complexity:.1f}")

# Efficiency vs Accuracy trade-off
print(f"\n⚖️ EFFICIENCY vs ACCURACY TRADE-OFF:")
print("=" * 45)

accuracies = {
    'Original Features': best_original_result['accuracy'],
    'PCA Features': best_pca_result['accuracy'], 
    'LDA Features': best_lda_result['accuracy']
}

for config_name in configs:
    accuracy = accuracies[config_name]
    complexity = complexity_scores[config_name]
    efficiency = accuracy / (complexity / 100)  # Higher is better
    
    print(f"{config_name}:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Complexity: {complexity:.1f}")
    print(f"  Efficiency Score: {efficiency:.4f}")
    print(f"  Recommendation: {'High efficiency' if efficiency > 0.02 else 'Standard efficiency'}")

print(f"\nSVM interpretability analysis complete!")
print(f"📈 Support vector patterns reveal model decision-making process")
print(f"🔍 Kernel choice significantly impacts interpretability and complexity")
print(f"⚖️ Trade-off between model accuracy and interpretability identified")

## 5. Conclusie

